# Clase 102 — Keras Sequential API

La **Sequential API** es la forma más simple de construir un modelo: una **pila
lineal** de capas donde la salida de una es la entrada de la siguiente. Perfecta
cuando no hay ramas, skip connections ni múltiples entradas/salidas.

Requiere: `tensorflow` / `keras` (≥ 3.0). Se ejecuta en Colab con GPU.

## 1. Dos sintaxis equivalentes: lista vs `.add()`

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

# (a) lista en el constructor
modelo_a = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

# (b) incremental con .add()
modelo_b = keras.Sequential()
modelo_b.add(keras.Input(shape=(784,)))
modelo_b.add(layers.Dense(128, activation="relu"))
modelo_b.add(layers.Dense(64, activation="relu"))
modelo_b.add(layers.Dense(10, activation="softmax"))

print("params (a):", modelo_a.count_params())
print("params (b):", modelo_b.count_params())
print("misma arquitectura:", modelo_a.count_params() == modelo_b.count_params())

## 2. `model.summary()`: leer la arquitectura

In [ ]:
modelo_a.summary()   # capas, output shape y # de parámetros trainable

## 3. Conteo de parámetros a mano

`Dense(n_out)` sobre entrada `n_in` tiene `n_in * n_out + n_out` parámetros (pesos + bias).

In [ ]:
capas = [(784, 128), (128, 64), (64, 10)]
total = 0
for n_in, n_out in capas:
    p = n_in * n_out + n_out
    total += p
    print(f"Dense({n_out}) sobre {n_in}: {n_in}*{n_out} + {n_out} = {p}")
print("total calculado a mano:", total)
print("total de Keras        :", modelo_a.count_params())

## 4. Ciclo completo: compile, fit, evaluate, predict

In [ ]:
(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = (X_tr.reshape(-1, 784).astype("float32")) / 255.0
X_te = (X_te.reshape(-1, 784).astype("float32")) / 255.0

modelo_a.compile(optimizer="adam",
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
modelo_a.fit(X_tr, y_tr, validation_split=0.1,
             epochs=5, batch_size=64, verbose=2)   # verbose=2: 1 línea/época
test_loss, test_acc = modelo_a.evaluate(X_te, y_te, verbose=0)
print(f"accuracy en test: {test_acc:.3f}")

probs = modelo_a.predict(X_te[:5], verbose=0)      # shape (5, 10)
print("clases predichas:", probs.argmax(axis=1))
print("clases reales   :", y_te[:5])

## 5. Guardar y restaurar con el formato `.keras`

In [ ]:
ruta = "modelo_fashion.keras"     # formato nativo Keras 3 (zip)
modelo_a.save(ruta)

modelo_cargado = keras.models.load_model(ruta)
p1 = modelo_a.predict(X_te[:10], verbose=0)
p2 = modelo_cargado.predict(X_te[:10], verbose=0)
print("predicciones idénticas tras recargar:", np.allclose(p1, p2))

## 6. Efecto del `batch_size`

In [ ]:
# batch grande = updates más estables pero menos updates por época
for bs in (32, 256):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    h = m.fit(X_tr, y_tr, epochs=3, batch_size=bs, verbose=0)
    print(f"batch_size={bs:>3}: accuracy final train = {h.history['accuracy'][-1]:.3f}")

## Ejercicios

1. **Dos sintaxis**: construí el mismo modelo con `Sequential([...])` y con
   `.add()` y verificá que `summary()` es idéntico.
2. **Conteo de parámetros**: para `Dense(128) -> Dense(64) -> Dense(10)` calculá
   los parámetros a mano y comparalos con `model.summary()`.
3. **Save/load**: entrená 5 épocas, guardá en `.keras`, recargá y verificá que
   `predict` da resultados idénticos.
4. **Tiny vs Medium vs Wide**: compará `Dense(64)->Dense(10)`,
   `Dense(256)->Dense(128)->Dense(10)` y `Dense(1024)->Dense(10)` en parámetros y
   accuracy. Comprobá que "profundo" gana a "solo ancho".

## Conclusiones

- **Sequential** modela una pila lineal de capas: simple y declarativa.
- `model.summary()` muestra shapes y parámetros por capa; el conteo se deriva de `n_in*n_out+n_out`.
- El ciclo estándar es **compile -> fit -> evaluate -> predict**.
- El formato **`.keras`** (Keras 3) guarda arquitectura + pesos + optimizador en un zip.
- Para topologías con ramas o multi-input/output, Sequential no alcanza: se usa la Functional API (clase 103).

## ✅ Soluciones de los ejercicios

Sequential API: dos sintaxis equivalentes, conteo de parámetros, guardado/carga `.keras`, formas en `predict` y niveles de `verbose`. Sin TF se validan por AST.

**Ej. 1 — Dos sintaxis.** `Sequential([...])` vs `add(...)` producen la misma arquitectura (mismo `count_params`).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_a():
    return keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")])

def build_b():
    m = keras.Sequential(); m.add(keras.Input((784,)))
    m.add(layers.Dense(128, activation="relu"))
    m.add(layers.Dense(64, activation="relu"))
    m.add(layers.Dense(10, activation="softmax"))
    return m

a, b = build_a(), build_b()
print("params A:", a.count_params(), "| params B:", b.count_params())
assert a.count_params() == b.count_params()
print("Ambas sintaxis construyen exactamente la misma red.")

**Ej. 2 — Conteo de parámetros.** Una Dense tiene `(in + 1) * out` parámetros (el +1 es el bias); lo verificamos contra `count_params()`.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128), layers.Dense(64), layers.Dense(10)])
manual = (784 + 1) * 128 + (128 + 1) * 64 + (64 + 1) * 10
print("calculo a mano:", manual, "| count_params():", model.count_params())
assert manual == model.count_params()
model.summary()

**Ej. 3 — Guardado/carga.** Tras entrenar, `save('m.keras')` + `load_model` reproduce `predict` idéntico.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.mnist.load_data()
Xtr = Xtr.reshape(-1, 784) / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.fit(Xtr, ytr, epochs=5, verbose=0)
model.save("m.keras")
reloaded = keras.models.load_model("m.keras")
assert np.allclose(model.predict(Xtr[:5], verbose=0), reloaded.predict(Xtr[:5], verbose=0))
print("predict identico tras save/load con el formato nativo .keras.")

**Ej. 4 — Predict batch vs individual.** Keras siempre espera la dimensión batch al frente: `(784,)` → `(1, 784)`.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([keras.Input((784,)), layers.Dense(10, activation="softmax")])
one = np.random.rand(784).astype("float32")
print("shape sin batch:", one.shape, "-> hay que expandir a (1, 784)")
p_one = model.predict(one[np.newaxis, :], verbose=0)             # (1, 10)
p_many = model.predict(np.random.rand(100, 784).astype("float32"), verbose=0)  # (100, 10)
print("predict individual:", p_one.shape, "| batch:", p_many.shape)
assert p_one.shape == (1, 10) and p_many.shape == (100, 10)

**Ej. 5 — Verbose.** `verbose=0` silencioso (ideal en CI/notebooks), `=1` barra por batch, `=2` una línea por época.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.mnist.load_data()
Xtr = Xtr.reshape(-1, 784) / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(32, activation="relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.fit(Xtr, ytr, epochs=1, verbose=0)   # nada
model.fit(Xtr, ytr, epochs=1, verbose=2)   # una linea por epoca
print("verbose: 0=silencioso, 1=barra de progreso, 2=una linea por epoca.")